# Proyek Akhir: Menyelesaikan Permasalahan Institusi Pendidikan
## Business Understanding

**Latar Belakang:**  
Jaya Jaya Institut adalah institusi pendidikan perguruan yang berdiri sejak tahun 2000. Meskipun telah mencetak banyak lulusan berprestasi, institusi ini menghadapi masalah serius: **tingginya angka dropout mahasiswa**.

Dropout yang tinggi berdampak pada:
- Reputasi institusi yang menurun
- Hilangnya potensi pendapatan dari mahasiswa yang tidak menyelesaikan studi
- Kegagalan institusi dalam memenuhi misi pendidikannya

**Permasalahan Bisnis:**
1. Faktor-faktor apa yang paling berkontribusi terhadap keputusan mahasiswa untuk dropout?
2. Bagaimana cara mendeteksi secara dini mahasiswa yang berpotensi dropout agar dapat diberikan bimbingan khusus?

**Cakupan Proyek:**
- Analisis eksplorasi data untuk memahami pola dropout
- Business dashboard untuk monitoring performa mahasiswa
- Model machine learning untuk prediksi dropout
- Prototype aplikasi Streamlit yang dapat diakses secara online

## Persiapan
### Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, ConfusionMatrixDisplay)
import pickle, json, warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
print('✓ Library berhasil diimport')

### Load Dataset

In [ ]:
df = pd.read_csv('data.csv', sep=';')
print(f'Shape: {df.shape}')
print(f'Kolom: {df.columns.tolist()}')
df.head()

## Data Understanding
Dataset berisi informasi 4.424 mahasiswa dengan 37 fitur mencakup:
- **Demografi**: usia, jenis kelamin, status pernikahan, kewarganegaraan
- **Latar belakang akademik**: kualifikasi sebelumnya, nilai masuk
- **Performa akademik**: unit kurikuler semester 1 & 2 (terdaftar, lulus, nilai)
- **Faktor sosial-ekonomi**: penerima beasiswa, status hutang, biaya kuliah
- **Target**: Status (Graduate / Dropout / Enrolled)

In [ ]:
# Distribusi target
status_counts = df['Status'].value_counts()
print('Distribusi Status:')
for k, v in status_counts.items():
    print(f'  {k}: {v} ({v/len(df)*100:.1f}%)')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colors = ['#2ecc71', '#e74c3c', '#3498db']
axes[0].pie(status_counts, labels=status_counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90)
axes[0].set_title('Proporsi Status Mahasiswa', fontweight='bold')

bars = axes[1].bar(status_counts.index, status_counts.values, color=colors, width=0.5)
for bar, val in zip(bars, status_counts):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+10,
                 str(val), ha='center', fontweight='bold')
axes[1].set_title('Jumlah Mahasiswa per Status', fontweight='bold')
axes[1].set_ylabel('Jumlah Mahasiswa')
plt.tight_layout()
plt.savefig('status_distribution.png', bbox_inches='tight')
plt.show()

In [ ]:
# Cek missing values dan duplikat
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Duplikat: {df.duplicated().sum()}')
df.describe().round(2)

## Data Preparation
Untuk pemodelan, kita fokus pada klasifikasi **Dropout vs Graduate** (exclude Enrolled) karena:
- Mahasiswa Enrolled belum memiliki outcome final
- Klasifikasi binary lebih actionable untuk HR

In [ ]:
# Filter hanya Dropout dan Graduate
df_model = df[df['Status'].isin(['Dropout', 'Graduate'])].copy()
df_model['Target'] = (df_model['Status'] == 'Dropout').astype(int)

print(f'Shape setelah filter: {df_model.shape}')
print(f'Dropout rate: {df_model["Target"].mean()*100:.1f}%')

feature_cols = [c for c in df_model.columns if c not in ['Status', 'Target']]
X = df_model[feature_cols]
y = df_model['Target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train: {X_train.shape[0]} | Test: {X_test.shape[0]}')

## Exploratory Data Analysis (EDA)

In [ ]:
# EDA 1: Dropout by key categorical features
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

features = {
    'Tuition_fees_up_to_date': {0: 'Tidak', 1: 'Ya'},
    'Scholarship_holder': {0: 'Tidak', 1: 'Ya'},
    'Debtor': {0: 'Tidak', 1: 'Ya'},
    'Gender': {0: 'Perempuan', 1: 'Laki-laki'}
}

for ax, (col, label_map) in zip(axes.flatten(), features.items()):
    ct = df_model.groupby([col, 'Target']).size().unstack(fill_value=0)
    ct.index = [label_map.get(i, str(i)) for i in ct.index]
    rate = (ct[1] / ct.sum(axis=1) * 100)
    bars = ax.bar(rate.index, rate, color=['#3498db','#e74c3c'][:len(rate)], width=0.4)
    for bar, v in zip(bars, rate):
        ax.text(bar.get_x()+bar.get_width()/2, v+0.5, f'{v:.1f}%',
                ha='center', fontweight='bold', fontsize=10)
    ax.set_title(f'Dropout Rate by {col}', fontweight='bold')
    ax.set_ylabel('Dropout Rate (%)')
    ax.set_ylim(0, 100)

plt.suptitle('Dropout Rate berdasarkan Faktor Kunci', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_categorical.png', bbox_inches='tight')
plt.show()

In [ ]:
# EDA 2: Distribusi performa akademik
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for col, ax, title in [
    ('Curricular_units_2nd_sem_approved', axes[0], 'Unit Lulus Sem 2'),
    ('Curricular_units_2nd_sem_grade',    axes[1], 'Nilai Rata-rata Sem 2')
]:
    for label, color, name in [(0,'#2ecc71','Graduate'),(1,'#e74c3c','Dropout')]:
        ax.hist(df_model[df_model['Target']==label][col], bins=20,
                alpha=0.65, color=color, label=name)
    ax.set_title(f'Distribusi {title} by Status', fontweight='bold')
    ax.set_xlabel(title)
    ax.set_ylabel('Jumlah Mahasiswa')
    ax.legend()

plt.tight_layout()
plt.savefig('eda_academic.png', bbox_inches='tight')
plt.show()

In [ ]:
# EDA 3: Dropout rate by Age Group
df_model['AgeGroup'] = pd.cut(df_model['Age_at_enrollment'],
    bins=[17,20,23,26,30,60], labels=['18-20','21-23','24-26','27-30','30+'])

age_rate = df_model.groupby('AgeGroup')['Target'].mean() * 100

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(age_rate.index.astype(str), age_rate, color='#e74c3c', alpha=0.8)
for bar, v in zip(bars, age_rate):
    ax.text(bar.get_x()+bar.get_width()/2, v+0.5, f'{v:.1f}%',
            ha='center', fontweight='bold')
ax.set_title('Dropout Rate berdasarkan Kelompok Usia', fontweight='bold')
ax.set_xlabel('Kelompok Usia')
ax.set_ylabel('Dropout Rate (%)')
plt.tight_layout()
plt.savefig('eda_age.png', bbox_inches='tight')
plt.show()

In [ ]:
# EDA 4: Correlation heatmap (top features)
top_cols = ['Curricular_units_2nd_sem_approved','Curricular_units_1st_sem_approved',
            'Curricular_units_2nd_sem_grade','Curricular_units_1st_sem_grade',
            'Tuition_fees_up_to_date','Scholarship_holder','Age_at_enrollment',
            'Debtor','Admission_grade','Target']

fig, ax = plt.subplots(figsize=(10, 8))
corr = df_model[top_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax)
ax.set_title('Correlation Matrix – Fitur Utama vs Target', fontweight='bold')
plt.tight_layout()
plt.savefig('eda_correlation.png', bbox_inches='tight')
plt.show()

## Modeling
Menggunakan **Random Forest** dengan `class_weight='balanced'` untuk menangani ketidakseimbangan kelas.

In [ ]:
import os
os.makedirs('model', exist_ok=True)

model = RandomForestClassifier(
    n_estimators=200, max_depth=10,
    random_state=42, class_weight='balanced', n_jobs=-1
)
model.fit(X_train_sc, y_train)

cv_scores = cross_val_score(model, X_train_sc, y_train, cv=5, scoring='roc_auc')
print(f'Cross-Val ROC-AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print('✓ Model berhasil dilatih')

## Evaluation

In [ ]:
y_pred  = model.predict(X_test_sc)
y_proba = model.predict_proba(X_test_sc)[:, 1]
roc     = roc_auc_score(y_test, y_proba)

print('=== Classification Report ===')
print(classification_report(y_test, y_pred, target_names=['Graduate','Dropout']))
print(f'ROC-AUC Score: {roc:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['Graduate','Dropout'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix', fontweight='bold')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[1].plot(fpr, tpr, color='#e74c3c', lw=2, label=f'ROC-AUC = {roc:.4f}')
axes[1].plot([0,1],[0,1],'k--',lw=1)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve', fontweight='bold')
axes[1].legend()
plt.tight_layout()
plt.savefig('model_evaluation.png', bbox_inches='tight')
plt.show()

In [ ]:
# Feature Importance
importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=True)
top15 = importances.tail(15)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#e74c3c' if i >= len(top15)-5 else '#3498db' for i in range(len(top15))]
ax.barh(top15.index, top15.values, color=colors)
ax.set_title('Top 15 Feature Importances', fontweight='bold')
ax.set_xlabel('Importance Score')
legend_patches = [mpatches.Patch(color='#e74c3c', label='Top 5'),
                  mpatches.Patch(color='#3498db', label='Lainnya')]
ax.legend(handles=legend_patches)
plt.tight_layout()
plt.savefig('feature_importance.png', bbox_inches='tight')
plt.show()

## Menyimpan Model

In [ ]:
with open('model/rf_dropout.pkl', 'wb') as f:
    pickle.dump(model, f)
with open('model/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

meta = {
    'feature_cols': feature_cols,
    'model_type': 'RandomForestClassifier',
    'roc_auc': round(roc, 4),
    'top_features': importances.tail(10).iloc[::-1].index.tolist()
}
with open('model/model_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print('✓ Model disimpan ke model/rf_dropout.pkl')
print('✓ Scaler disimpan ke model/scaler.pkl')
print('✓ Metadata disimpan ke model/model_meta.json')

## Conclusion

**Jawaban atas Permasalahan Bisnis:**

**1. Faktor utama yang mempengaruhi dropout:**
- **Performa akademik** adalah prediktor terkuat: mahasiswa dengan unit kurikuler lulus (approved) yang rendah di semester 1 & 2 memiliki risiko dropout sangat tinggi
- **Nilai rata-rata** semester 1 & 2 yang rendah berkorelasi kuat dengan dropout
- **Status pembayaran SPP** (Tuition_fees_up_to_date): mahasiswa yang menunggak pembayaran jauh lebih berisiko dropout
- **Penerima beasiswa** memiliki dropout rate lebih rendah karena motivasi finansial
- **Usia saat mendaftar**: mahasiswa yang lebih tua (>26 tahun) cenderung lebih berisiko dropout
- **Status debitur**: mahasiswa yang memiliki hutang akademik memiliki dropout rate lebih tinggi

**2. Model Prediksi:**
- Model Random Forest mencapai **ROC-AUC 0.9695** dan **akurasi 92%**
- Model ini sangat handal untuk mengidentifikasi mahasiswa berisiko dropout sejak dini

**Rekomendasi Action Items:**
1. **Monitoring akademik semester 1** – Identifikasi mahasiswa dengan <3 unit lulus di semester 1 untuk intervensi dini
2. **Program beasiswa tepat sasaran** – Prioritaskan mahasiswa berpotensi dropout dengan kesulitan finansial
3. **Sistem peringatan dini** – Implementasikan prototype Streamlit untuk prediksi rutin setiap semester
4. **Konseling akademik** – Wajibkan sesi konseling bagi mahasiswa dengan nilai rata-rata <10
5. **Fleksibilitas pembayaran** – Tawarkan cicilan bagi mahasiswa yang menunggak SPP sebelum mereka memutuskan dropout